<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
# Setting up a relative binding free energy (RBFE) calculation

 This notebook demonstrates the **setup stage** of a Free Energy Perturbation (FEP) workflow using the [OpenFE](https://openfreeenergy.org) toolkit.  

Specifically:
- Select two ligands from a TYK2 benchmark set (loaded from SDF).
- Define chemical components for the protein, ligands, and solvent.
- Generate `Atom mappings` between ligands using `LoMap`.
- Define bound (complex) and unbound (solvent) `Chemical systems`.
- Specify a relative hybrid topology `protocol` and create `Transformation` objects for both legs of the TD cycle.
- Serialize these transformations to JSON for later (possibly external) execution.
- Show how results would be gathered into a final $\Delta \Delta G$ estimate (assuming simulations were run externally), using MBAR


**Limitations of this notebook:**  
- This notebook covers **setup only**. Simulation execution (sampling at multiple λ windows), free energy estimation (MBAR/BAR), and convergence analysis will be handled in a separate script.  
- Protocol settings here use OpenFE defaults. In production, one should explicitly specify λ schedules, equilibration/production lengths, PME and softcore parameters, and random seeds.  
- Input preparation assumes ligands have consistent protonation/tautomeric states and charges.

</div>


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

# Free Energy Perturbation and Thermodynamic Integration

**Why use FEP?**  
Free Energy Perturbation is a rigorous statistical mechanics method to compute the relative binding free energy $\Delta \Delta G$ between two ligands or between different states of a molecular system. In drug discovery, it helps predict how chemical modifications affect binding affinity, enabling more accurate prioritization of compounds than simpler scoring functions or docking approaches. FEP is particularly useful when evaluating small structural changes (substituents, scaffold hops, charge changes) where other methods may fail.

**How does FEP work?**  
FEP relies on simulating two molecular states (e.g., ligand A bound to a protein vs. ligand B bound to the protein) and computing the free energy difference via *alchemical transformations*. This is achieved by gradually “morphing” one molecule into another (i.e., one Hamiltonian $U_A$ into another $U_B$ using a coupling parameter (λ) in molecular dynamics simulations $$U(\lambda) = (1-\lambda)U_A + \lambda U_B, 0 \leq \lambda \leq 1. $$ Simulation are run at each lambda step, and then integrated over these intermediate states using **MBAR** to remove the bias, the relative free energy difference ΔΔG can be obtained. A so called *FEP campaign* involves exploring all pairs of ligands that are available, via defining possibly an Alchemical Network.

**In practice:**  
1. Define the two molecular states (ligand A → ligand B).  
2. Set up alchemical intermediates along the λ coordinate. This requires a mapping between atoms in A to atoms in B, it is a smooth mathematical transformation, not a physical one
3. Run molecular dynamics (or Monte Carlo) simulations for each λ state.  
4. Collect energy statistics from each $\lambda$ state and compute $\Delta G$ via exponential averaging (Zwanzig relation) or free energy estimators (BAR/MBAR).  

This notebook demonstrates how to set up and analyze such calculations in practice.
</div>

In [ ]:
import matplotlib, os
%matplotlib inline

import numpy as np
import openfe      # this was installed using Miniforge, by following the steps indicated on https://docs.openfree.energy/en/latest/installation.html

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Loading the players: here we just focus on a single transformation A --> B between TYK2 ligands

In [ ]:
from openff.toolkit import Molecule
from openfe import SmallMoleculeComponent, SolventComponent, ProteinComponent

protein_file = './tyk2_protein.pdb'
ligands_file = './tyk2_ligands.sdf'    # SDF molfile format, containing multiple molecules sequentially (separated by ####)

# ===== protein
protein = ProteinComponent.from_pdb_file(protein_file, name = 'TYK2')

# ===== solvent
#solvent = SolventComponent(positive_ion='Na', negative_ion='Cl',
#                           neutralize=True, ion_concentration=0.15*unit.molar)
solvent = SolventComponent()    # standard solvnet

# ===== Load ligands using OpenFF toolkit
ligands_sdf = Molecule.from_file(ligands_file)
ligand_mols = [SmallMoleculeComponent.from_openff(sdf) for sdf in ligands_sdf]

# we can get the SMILES representation printed out to screeb
print(f'SMILES representation of first ligand = {ligand_mols[0].smiles}')
print(f'SMILES representation of second ligand = {ligand_mols[1].smiles}')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Define an atomic mapping from ligand 1 to ligand 2

In the [hybrid topology RBFE protocol](https://docs.openfree.energy/en/latest/guide/protocols/relativehybridtopology.html) , an atom mapping defines which atoms are mutated during the alchemical transformation. Mapping relies only on chemical graph connectivity (and sometimes 3D geometry), so they only need atom identities and bonding

Two popular options involve:
1. `LomapAtomMapper`
    * based on the maximum common substructure (MCS)
2. `KartografAtomMapper`
    * based on the 3D geometries of the ligands

While we use the defaults here, please note that the various supported arguments of `Lomap` and `Kartograf` can be passed to the atom mapper.

In [ ]:
# choose a mapping strategy and apply that to the one ligand pair that we have
from openfe.setup import LomapAtomMapper

mapper = LomapAtomMapper()
lomap_mapping = next(mapper.suggest_mappings(ligand_mols[0], ligand_mols[1]))     # "next" is necessary for visaulization

In [ ]:
# print informatioon about the mapping, so we can see which atoms go into which
print(f'Ligand A = {lomap_mapping.componentA}')
print(f'Ligand B = {lomap_mapping.componentB}')

print('Explicit Atom Mapping = ')
print(lomap_mapping.componentA_to_componentB)      # these attributes are also printed out for each edge of a hypothetical alchemic network

<div style="background-color: lightyellow; padding: 10px; border-radius: 5px;">

### ========= Interlude: visualization helper ==========

We can also visualize the atom mappings by invoking the individual OpenFE `AtomMapping` objects directly.

Unique atoms between each mapping are shown in red, and atoms which are mapped but undergo element changes are shown in blue. Bonds which either involve atoms that are unique or undergo element changes are highlighted in red.

In [ ]:
from IPython.display import display, HTML, Image, Markdown, Latex
lomap_mapping

<div style="background-color: lightyellow; padding: 10px; border-radius: 5px;">

It is also possible to visualize the mapping in 3D using the embedded `py3dmol`:

Here, the visualization_3D method displays the two end state molecules (left and right), in addition to the hybrid molecule (middle).

Atoms that have the same sphere color in both end states are mapped (i.e. will be interpolated between each other), whilst those that do not have a coloured sphere are unmapped (i.e. will be transformed into dummy atoms in the opposite end state).

In [ ]:
# Visualize the mapping in 3D
from openfe.utils import visualization_3D

visualization_3D.view_mapping_3d(lomap_mapping, show_atomIDs=True)

#### Schematic of a Thermodynamic Cycle underlying FEP calculations

<img src="rbfe_thermocycle.png" alt="TYK2 ligand overlay" width="400">

The solvent-only calculation works as a reference leg that ensures only the protein-binding component of the free energy survives.

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Alchemical transformation, 1.0: define `ChemicalSystem`s 

`ChemicalSystems` take in:

1. A dictionary of the chemical components (e.g. `SmallMoleculeComponent`,
   `ProteinComponent`, `SolventComponent`) defining the system.

2. An identifier name (optional), for the `ChemicalSystem`. This is used as part
   of the hash identifier of the `ChemicalSystem`, and can help distinguish between
   otherwise comparable systems.


In the case of a relative ligand binding free energy calculation for ligand_A -> ligand_B, four `ChemicalSystems` must be defined, so that both legs of the Thermodynamic Cycle are there: both ligands in solvent with/out protein.

In [ ]:
# ligands that we will connect by an alchemical transformation for FEP
from openfe import ChemicalSystem

A_complex = ChemicalSystem({'ligand' : ligand_mols[0],
                            'solvent' : solvent,
                            'protein' : protein}, name = 'complex_A')

B_complex = ChemicalSystem({'ligand' : ligand_mols[1],
                            'solvent' : solvent,
                            'protein' : protein}, name = 'complex_B')

A_solvent = ChemicalSystem({'ligand' : ligand_mols[0],
                            'solvent' : solvent
                            }, name = 'solvent_A')

B_solvent = ChemicalSystem({'ligand' : ligand_mols[1],
                            'solvent' : solvent
                            }, name = 'solvent_B')

print(A_complex, B_solvent)

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Alchemical Transformation, 2.0: define simulation `Protocol`: simulation setup, $\lambda$ schedule, equilibration...

Now that we have a set of atom mappings defined, we know which atoms should
undergo alchemical transformations to capture the free energy cost of
transforming from one ligand to another.

To simulate this transformation, we use the equilibrium RBFE protocol
implemented in OpenFE. This uses `OpenMM` to run a Perses-like relative
ligand binding free energy calculation using a single topology approach.

To achieve this simulation, the following steps need to happen:

1. Create OpenMM systems of both end states, including charging the ligands (`partial_charge_settings`)

2. Create a hybrid topology based on these defined end states

3. Set an appropriate $\lambda$ schedule

4. Set a MultiState reporter to write out appropriate coordinates and energies

5. Create an OpenMM sampler (in this case we will be using a replica exchange sampler)

6. Carry out the necessary simulation steps (minimization, equilibration, and production)


The `RelativeHybridTopologyProtocol` class in `openfe.protocols.openmm_rfe`
implements a means to achieve all the above with minimal intervention. Please note that a protocol is an abstract configuration choice, it still does not inherit information from the ligands and protein we would like to simulate

Here we work through its usage for the `ligand1` -> `ligand2` binding free energy
test case. As this involves both a relative binding free energy in solvent
and complex phases, `RelativeHybridTopologyProtocol` will
be used to build two separate `ProtocolDAG` (directed-acyclic-graph) classes, one for each phase.
These `DAG`s (which contain the necessary individual simulations), are then executed to yield
the desired free energy results.

**Note**: 
- just for the purpose of setups, we are retaining some fo the standard simulation settings, while showing some options
- This is a good time to **charge** the ligands.
- ligand charges are not tabulated, chemistry there is too diverse
- charges are still necessary for the electrostatic interactions of the force field

In [ ]:
# define protocol
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol
from openff.units import unit

all_options = False

if all_options == True:
    # this is printing out all the options that can be modified withint the configuration/protocol file
    s = RelativeHybridTopologyProtocol.default_settings()
    print(getattr(s, "model_dump", getattr(s, "dict"))())

# Create the default settings, print simulation steps, force fields and everything
rbfe_settings = RelativeHybridTopologyProtocol.default_settings()
print(rbfe_settings.simulation_settings)

# edit some parameters
rbfe_settings.thermo_settings.temperature = 300.0 * unit.kelvin
rbfe_settings.thermo_settings.pressure = 1.0 * unit.atmosphere
rbfe_settings.integrator_settings.timestep = 2.0 * unit.femtoseconds
#rbfe_settings.integrator_settings.langevin_friction = 1.0 / unit.picosecond
rbfe_settings.integrator_settings.barostat_frequency = 25  # every 50 fs if dt=2 fs


# Sampling lengths
rbfe_settings.simulation_settings.equilibration_length = 0.2 * unit.nanosecond
rbfe_settings.simulation_settings.production_length    = 1.0 * unit.nanosecond

#rbfe_settings.equilibration_length = 2   # 200 ps @ 2 fs
#rbfe_settings.production_steps    = 500000   # 1 ns per window (tune as needed)
#rbfe_settings.minimization_steps  = 5000

# Seeds for reproducibility
#rbfe_settings.random_seed = 20250819


# Create RBFE Protocol class
rbfe_protocol = RelativeHybridTopologyProtocol(
    settings=rbfe_settings
)
print(type(rbfe_protocol))    # openmm protocol for optimization

# This Protocol defines the procedure to estimate a free energy difference between two chemical systems,
# with the details of the two end states yet to be defined.

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Complete `Transformation`: combine `ChemicalSystems`, `mapping`, and `protocol`

A transformation can be dumped into a .json file ('configuration file'), so that it can be run outside of the notebook from the command line, or be farmed out as individual jobs on a HPC cluster. so that they can be run independently using the `openfe quickrun` command.

As previously detailed, we create two sets of transformations for the complex and the solvent legs of the thermodynamic cycle.

In [ ]:
# transformation definition
from openfe import Transformation
transformation_complex = Transformation(stateA = A_complex, stateB = B_complex,
                                        mapping = lomap_mapping, protocol = rbfe_protocol, name = 'complex_transf')

transformation_solvent = Transformation(stateA = A_solvent, stateB = B_solvent,
                                        mapping = lomap_mapping, protocol = rbfe_protocol, name = 'solvent_transf')

save_ = True

if save_ == True:

    folder = 'LB_tests'
    
    if not os.path.exists(folder):
        os.makedirs(folder)   # creates intermediate dirs if needed
        print(f"Created folder: {folder}")
    else:
        print(f"Folder already exists: {folder}")

    transformation_complex.dump(folder + 'Test_transformationAB_complex.json')
    transformation_solvent.dump(folder + 'Test_transformationAB_solvent.json')

To summarize, this `Transformation` contains:
- chemical models of both sides of the alchemical transformation in `stateA` and `stateB`
- the correspondence of items in these two sides in `mapping`
- a description of the exact computational algorithm to use to perform the estimate in `protocol`

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Run `Transformations` as molecular simulations: 

With the Transformation defined, we can move onto creating the `ProtocolDAG`.

The `Transformation.create()` method creates a directed-acyclic-graph (DAG) of computational tasks necessary for creating an estimate of the free energy difference between the two chemical systems. `execute_DAG` allows to execute and run the transformation

The individual pieces of computational work are called `Units`. In this particular `Protocol`, the Units defined are three independent repeats of the alchemical transformation

In [ ]:
# from here, Python API or others, save info
from openfe import execute_DAG

complex_dag = transformation_complex.create()
solvent_dag = transformation_solvent.create()

# Finally we can run the simulations...this is a code snippet, simulations have to be run on a HPC
# First, complex (bound) leg of the transformation
complex_folder = './complex'
os.mkdir(complex_folder)
complex_dag_results = execute_DAG(complex_dag, scratch_basedir=complex_path, shared_basedir=complex_path)

# Next the solvent state transformation
solvent_folder = './solvent'
os.mkdir(solvent_folder)
solvent_dag_results = execute_DAG(solvent_dag, scratch_basedir=solvent_path, shared_basedir=solvent_path)

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

# Analyze the results. 

There is a built in CLI command `openfe gather` to automatically combine all the simulation results and output the $\Delta \Delta G$ estimate with the uncertainty coming from three independent productiion runs. Here, we can just use the `gather()` methods

Results are available for a much larger Alchemical Network, which contains our standard A --> B transformation. So we can procure those results and take it from there.

Standard **Multistate Bennett Acceptance Ratio (MBAR)** (histogram free improvement over **WHAM**, which also extends to any type of TD state, beyond umbrella sampling enhanced sampling) is used to pool together and reweigh free energy values from different TD states (i.e., different $\lambda$) so to get an unbiased result.

Assume we have:
* $K$ states, each corresponding to a $\lambda_k$ value; so, reduced potentials: $u_k(x) = \beta U_k (x)$
*  $N_k$ is the total number of samples for state, so that $\sum_k N_k = N$
* Configurations $x_n, n = 1,\ldots, N$, pooled from all states

The unknown dimensionless free energies $\hat{f}_i$ (defined up to an additive constant) are approximated by solving the MBAR self-consistent equations:
$$\hat{f}_i = -\ln \left[ \sum_{n=1}^N \frac{e^{-u_i(x_n)}}{\sum_k N_k e^{-u_k(x_n) + f_k}}\right]$$

Please note the nested sums over the states and the configurations; Once **free energies** are solved for, then we can reweigh configuration $x_n$ in state $i$

$$W_{ni} = \frac{e^{-u_i(x_n) + \hat{f}_i}}{\sum_k^K N_k e^{-u_k(x_n) + \hat{f}_k}}$$

In RBFE workflows, compute $\Delta G_{ij} = -\frac{1}{\beta} \Delta f_{ij}$ for the alchemical path in the bound leg and the solvent leg, then take the cycle different for the $\Delta \Delta G$

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Analysis of the results

(These functions demonstrate how results would be combined; in this notebook no trajectories were actually run)

TO-DO select the relevant trajectories for this transformation only and provide the results

In [ ]:
# Upon running the simulations, the different transformations/legs of the TD cycle, then use built-in functions to get the energy estimate

# Get the complex and solvent results
complex_results = rbfe_protocol.gather([complex_dag_results])
solvent_results = rbfe_protocol.gather([solvent_dag_results])

print(f"Complex dG: {complex_results.get_estimate()}, err {complex_results.get_uncertainty()}")
print(f"Solvent dG: {solvent_results.get_estimate()}, err {solvent_results.get_uncertainty()}")

print(f"The Delta Delta G associated with the process is {complex_results.get_estimate() - solvent_results.get_estimate()}, up to uncertainty")

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Limitations & Next Steps
- This notebook covers only the setup stage of an RBFE calculation.  
- Running the MD simulations and sampling is computationally intensive and typically requires HPC resources.  
- The next step would be to propagate the system, collect ensembles, and use the OpenFE analysis toolkit to estimate ΔΔG with error bars.  
- Including such results was outside the scope of this learning exercise, but the workflow structure is already in place.

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### No Python API

All this preparation can be performed and customized wiht the [`OpenFE` CLI](https://docs.openfree.energy/en/stable/tutorials/rbfe_cli_tutorial.html). Assuming we set out to perform a full **campaign** (using all the ligans in the input) based on a network, this is how it would look like. `.yaml` is the extension of an additional setup file to be customized

`openfe plan-rbfe-network -M tyk2_ligands.sdf -p tyk2_protein.pdb -o network_setup --n-protocol-repeats 1 -s settings.yaml`